# **Machine Learning Project PART 2 | Step 1: Data Pre processing- ATU Winter 2024**  
**Author**: Lais Coletta Pereira  
**Lecturer**: Brian McGinley  

## Data Pre-processing
The first task will be to build a dataset from the annotated data, here are the steps taken in this notebook:

1. Extract Spectrogram for Each Call:
   - For each annotated call, extract a spectrogram and store it as a raw 2D array (not as an image).
   - Use the frequency range specified in the annotations to ensure spectrograms are aligned in terms of time and frequency.
   - Save metadata with each spectrogram (e.g., annotation, category, frequency, time range).

2. Pre-Processing for Longest and Broadest Spectrogram:
   - Identify the longest call (by duration) and the broadest (by frequency) call.
   - Use these to set baseline spectrogram dimensions, ensuring consistency across all spectrograms.

3. Saving Spectrograms as 2D Arrays:
   - Processed spectrograms will be saved as `.npz` files (which can store both the spectrogram and metadata) in their respective directories.

4. Create Spectrograms for "No-Call" Segments:
   - Spectrograms must be created for time periods without annotated calls, as these represent "no-calls."
   - These "no-call" spectrograms should have the same frequency range as the call spectrograms, ensuring dataset consistency.

5. Metadata Storage:
   - For each spectrogram, associated metadata (e.g., annotation, call type, start and end times, frequency range) should be stored alongside it.

6. Ensure Uniform Spectrogram Size:
   - Spectrograms will be resized (padded or cropped) to match the baseline dimensions based on the longest time duration and broadest frequency range.

This process ensures that all spectrograms are uniform in size, and both calls and "no-calls" are processed and categorized correctly with their respective metadata.


In [1]:
#Imports
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal

In [2]:
# Define output directory for spectrogram files and create it if not exists
output_dir = 'spectrogram_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

***Function to load audio and annotations, reads .wav and .txt annotation files:***

In [3]:
def load_audio_and_annotation(file_base_name):
    # Load the WAV audio file
    audio_path = file_base_name + '.wav'
    sample_rate, samples = wavfile.read(audio_path)
    
    # Load the annotation file (typically .txt or .csv)
    annot_file_path = file_base_name + '.Table.1.selections.txt'
    df = pd.read_csv(annot_file_path, sep='\t')
    
    return sample_rate, samples, df

**Function to extract audio segments based on annotations**

Extract audio segments based on start and end times from annotations, storing each segment along with its relevant information in a list.

In [4]:
def extract_audio_segments(samples, sample_rate, annotations):
    segments = []
    for _, row in annotations.iterrows():
        start_time, end_time = row['Begin Time (s)'], row['End Time (s)']
        start_sample, end_sample = int(start_time * sample_rate), int(end_time * sample_rate)
        segment = samples[start_sample:end_sample]
        segments.append({
            'Selection': row['Selection'],
            'Annotation': row['Annotation'],
            'Begin Time (s)': start_time,
            'End Time (s)': end_time,
            'audio_segment': segment,
            'Low Freq (Hz)': row['Low Freq (Hz)'],
            'High Freq (Hz)': row['High Freq (Hz)']
        })
    return segments


**Define a function to generate a spectrogram from the audio segment**

This function generates a spectrogram from an audio segment by applying the Short-Time Fourier Transform (STFT) using [scipy.signal.spectrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html). 
It allows us to specify the frequency range (fmin, fmax) and other parameters like window length, FFT points, and overlap to control the resolution of the spectrogram. The function returns frequency bins, time bins, and the spectrogram data (2D array). I am using the spectrogram.ipynp as a source for this part.


In [5]:
# Reference code https://stackoverflow.com/questions/38015319/how-to-create-a-numpy-array-from-a-pydub-audiosegment
def generate_spectrogram(audio_segment, sample_rate, fmin=20, fmax=1000, nperseg=2456, nfft=4096, noverlap=1228):
    """
    Generate spectrogram for a given audio segment based on frequency range (fmin, fmax).
    
    Parameters:
        - audio_segment (np.array): The raw audio data.
        - sample_rate (int): The sampling rate of the audio data.
        - fmin (int): The minimum frequency for the spectrogram. Default is 20 Hz.
        - fmax (int): The maximum frequency for the spectrogram. Default is 1000 Hz.
        - nperseg (int): The length of each segment for the FFT. Default is 2456 samples.
        - nfft (int): The number of FFT points. Default is 4096.
        - noverlap (int): The number of points to overlap between segments. Default is 1228.

    Returns:
        - frequencies (np.array): Frequency bins for the spectrogram.
        - times (np.array): Time bins for the spectrogram.
        - spectrogram_data (np.array): 2D spectrogram array.
    """
    # Generate the spectrogram using scipy.signal.spectrogram
    frequencies, times, spectrogram_data = signal.spectrogram(audio_segment, sample_rate, 
                                                              nperseg=nperseg, nfft=nfft, 
                                                              noverlap=noverlap, window='hann')
    
    # Threshold tiny values in the spectrogram (to avoid plotting issues) 
    spectrogram_data[spectrogram_data < 0.001] = 0.001
    
    # Slice the frequencies based on the desired range (fmin to fmax)
    freq_slice = np.where((frequencies >= fmin) & (frequencies <= fmax))
    frequencies = frequencies[freq_slice]
    spectrogram_data = spectrogram_data[freq_slice, :]
    
    return frequencies, times, spectrogram_data

***Generate Spectrogram***

In [6]:
def generate_spectrogram(audio_segment, sample_rate, fmin=20, fmax=1000, nperseg=2456, nfft=4096, noverlap=1228):
    frequencies, times, spectrogram_data = signal.spectrogram(
        audio_segment, sample_rate, nperseg=nperseg, nfft=nfft, noverlap=noverlap, window='hann')
    spectrogram_data[spectrogram_data < 0.001] = 0.001
    freq_slice = np.where((frequencies >= fmin) & (frequencies <= fmax))
    frequencies, spectrogram_data = frequencies[freq_slice], spectrogram_data[freq_slice, :]
    return frequencies, times, spectrogram_data


***Calculate Baseline Parameters***
1. Iterate through folders:  
   The function loops through each folder, list and reads the annotation files (those ending with `.Table.1.selections.txt`).

2. Read Annotations:  
   For each annotation file, the function calculates the duration of each segment by subtracting the "Begin Time" from the "End Time". It also retrieves the minimum and maximum frequency ranges for each annotation.

3. Update Maximum Duration and Frequency Range:  
   As each annotation file is processed, the function keeps track of:
   - The largest segment duration (`max_duration`).
   - The full frequency range (`max_freq_range`) encountered in the dataset.

4. Final Result:  
   After processing all the annotation files, the function assigns the values of `max_duration` and `max_freq_range` to the global variables `baseline_time` and `baseline_freq`.  
   These values are used in further operations like padding or cropping spectrograms to ensure that all spectrograms have consistent dimensions.


In [7]:
def calculate_baselines(folder_paths):
    # Initialize global variables to store the maximum segment duration and frequency range
    global baseline_time, baseline_freq
    
    # Initialize variables to track the maximum duration and frequency range across all files
    max_duration, max_freq_range = 0, (float('inf'), 0)
    
    # Iterate through each folder in the folder_paths list
    for folder_path in folder_paths:
        # Loop through each file in the current folder
        for file in os.listdir(folder_path):
            # Only process files that end with '.Table.1.selections.txt' (annotation files)
            if file.endswith('.Table.1.selections.txt'):
                
                # Load the annotation file as a dataframe
                df = pd.read_csv(os.path.join(folder_path, file), sep='\t')
                
                # Calculate the duration of each segment (End Time - Begin Time)
                durations = df['End Time (s)'] - df['Begin Time (s)']
                
                # Update the maximum duration encountered so far
                max_duration = max(max_duration, durations.max())
                
                # Get the minimum and maximum frequency values for the current file
                low_freq, high_freq = df['Low Freq (Hz)'].min(), df['High Freq (Hz)'].max()
                
                # Update the frequency range to include the current file's frequency values
                max_freq_range = (min(max_freq_range[0], low_freq), max(max_freq_range[1], high_freq))
    
    # Set the global baseline_time and baseline_freq with the maximum duration and frequency range
    baseline_time, baseline_freq = max_duration, max_freq_range


***Padding or Cropping Spectrograms***

This function ensures that a spectrogram has a consistent shape by either padding or cropping along both time and frequency axes. It calculates the desired dimensions based on a baseline time and frequency range, which are derived from the longest call in time and the broadest frequency range across all the segments. These values become the baseline for the largest spectrogram. The function then adjusts the spectrogram's time bins to match the maximum duration, and ensures the correct frequency bins are present by resizing according to the broadest frequency range. It uses NumPy operations like np.squeeze, np.pad, and np.reshape to adjust the spectrogram as needed.

In [8]:
def pad_or_crop_spectrogram(spectrogram_data, sample_rate, max_time_bins=2000):
    # Remove singleton dimensions if they exist (e.g., a 1D spectrogram)
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.squeeze.html
    spectrogram_data = np.squeeze(spectrogram_data)
    
    # If the spectrogram has fewer than 2 dimensions, reshape it into a 2D array (1 x n)
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.reshape.html
    if len(spectrogram_data.shape) < 2:
        spectrogram_data = spectrogram_data.reshape(-1, 1)

    # If the spectrogram is still 1D, reshape it into 2D: (1, n)
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.reshape.html
    if len(spectrogram_data.shape) == 1:
        spectrogram_data = spectrogram_data.reshape(1, -1)

    # Retrieve the current number of frequency bins and time bins from the spectrogram shape
    current_frequency_bins, current_time_bins = spectrogram_data.shape

    # Calculate the target number of time bins based on baseline duration and sample rate
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.linspace.html
    target_time_bins = int(baseline_time * sample_rate / spectrogram_data.shape[0])
    
    # Set the maximum time bins to `max_time_bins` (limit the maximum size)
    target_time_bins = min(target_time_bins, max_time_bins)

    # Padding for the time dimension (add zeros to match the target_time_bins)
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.pad.html
    padding_width = target_time_bins - current_time_bins
    if padding_width > 0:
        # If padding is needed, add zeros along the time axis
        spectrogram_data = np.pad(
            spectrogram_data,
            pad_width=((0, 0), (0, padding_width)),
            mode='constant', constant_values=0
        )
    elif current_time_bins > target_time_bins:
        # If the spectrogram is too large, crop it to the target size
        spectrogram_data = spectrogram_data[:, :target_time_bins]

    # Frequency axis resizing if necessary
    # Calculate the number of frequency bins based on baseline frequency range
    target_frequency_bins = len(np.linspace(baseline_freq[0], baseline_freq[1], spectrogram_data.shape[0]))
    
    current_frequency_bins = spectrogram_data.shape[0]
    if current_frequency_bins < target_frequency_bins:
        # If there are fewer frequency bins than required, add zero padding
        frequency_padding = target_frequency_bins - current_frequency_bins
        spectrogram_data = np.pad(
            spectrogram_data,
            pad_width=((frequency_padding, 0), (0, 0)),
            mode='constant', constant_values=0
        )
    elif current_frequency_bins > target_frequency_bins:
        # If there are more frequency bins than required, crop the spectrogram
        spectrogram_data = spectrogram_data[:target_frequency_bins, :]

    # Return the padded or cropped spectrogram
    return spectrogram_data


***Create "No Call" Segments***

Identify the periods where no calls are made by analyzing the gaps between annotated call segments. It extracts the "no-call" portions from the audio samples and returns them as separate segments. The function handles any remaining no-call period after the last annotation and returns a list of the segmented intervals with no calls.

In [9]:
def create_no_call_segments(samples, sample_rate, annotations):
    # Initialize an empty list to store the 'no call' segments 
    #Source: https://stackoverflow.com/questions/50310527/accessing-table-view-cell-information-from-another-view-controller-in-swift/50312258#50312258
    no_call_segments = []
    # Track the last end time of an annotation to determine the gaps between calls
    last_end_time = 0
    
    # Iterate over each row (each annotation) in the annotations DataFrame
    for _, row in annotations.iterrows():
        start_time = row['Begin Time (s)']
        # If there is a gap (the start of the current annotation is after the last end time)
        if start_time > last_end_time:
            # Extract the segment of audio between the previous annotation end and the current annotation start
            no_call_segment = samples[int(last_end_time * sample_rate):int(start_time * sample_rate)]
            no_call_segments.append(no_call_segment)
        
        # Update the last_end_time to be the later of the current or previous end time
        last_end_time = max(last_end_time, row['End Time (s)'])
    
    # If the last annotation doesn't cover the entire audio, add the remaining 'no-call' segment
    if last_end_time < len(samples) / sample_rate:
        no_call_segment = samples[int(last_end_time * sample_rate):]
        no_call_segments.append(no_call_segment)
    
    # Return the list of extracted 'no-call' segments
    return no_call_segments


***Helper Function for Categorization***

Determines the category name based on the folder path. For example, files in the folder 'Guttural rupe' are categorized as 'guttural_rupe'.

In [10]:
def get_category_name(folder_path):
    if 'Guttural rupe' in folder_path:
        return 'guttural_rupe'
    elif 'Rupes A and B' in folder_path:
        return 'rupe_A_and_B'
    elif 'Moan' in folder_path:
        return 'moan'
    else:
        return 'unknown_category'

Load files, process audio, generate spectrograms, padding/cropping and saving the data:

In [11]:
folder_paths = [
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Guttural rupe',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Rupes A and B',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Moan'
]

calculate_baselines(folder_paths)
dataset = []

# Extracting segments
for folder_path in folder_paths:
    category = get_category_name(folder_path)  # Get the category based on the folder path

    # Create subfolder for the category if it doesn't exist
    category_output_dir = os.path.join(output_dir, category)
    if not os.path.exists(category_output_dir):
        os.makedirs(category_output_dir)

    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_base_name = os.path.splitext(file)[0]
            file_base_path = os.path.join(folder_path, file_base_name)
            sample_rate, samples, df = load_audio_and_annotation(file_base_path)
            segments = extract_audio_segments(samples, sample_rate, df)

            # Add the category column to each segment dataframe
            for segment in segments:
                segment['Category'] = category
            dataset.append(pd.DataFrame(segments))

final_df = pd.concat(dataset, ignore_index=True)

# --- Generate and save spectrograms for calls ---
for index, segment_info in final_df.iterrows():
    audio_segment = segment_info['audio_segment']
    frequencies, times, spectrogram_data = generate_spectrogram(
        audio_segment, sample_rate, fmin=segment_info['Low Freq (Hz)'], fmax=segment_info['High Freq (Hz)']
    )

    # Apply padding or cropping based on the baseline
    spectrogram_data = pad_or_crop_spectrogram(spectrogram_data, sample_rate)

    metadata = {
        'Selection': segment_info['Selection'],
        'Annotation': segment_info['Annotation'],
        'Category': segment_info['Category'],
        'Spectrogram Shape': spectrogram_data.shape
    }
    
    # Save the file in the corresponding subfolder
    np.savez(
        os.path.join(category_output_dir, f"{segment_info['Selection']}_spectrogram.npz"),
        spectrogram=spectrogram_data, metadata=metadata
    )

# --- Generate and save no-call spectrograms --- 
no_call_segments = create_no_call_segments(samples, sample_rate, final_df)
no_calls_output_dir = os.path.join(output_dir, "no_calls")
if not os.path.exists(no_calls_output_dir):
    os.makedirs(no_calls_output_dir)

for idx, no_call_segment in enumerate(no_call_segments):
    frequencies, times, spectrogram_data = generate_spectrogram(
        no_call_segment, sample_rate, fmin=baseline_freq[0], fmax=baseline_freq[1]
    )

    # Apply padding or cropping based on the baseline
    spectrogram_data = pad_or_crop_spectrogram(spectrogram_data, sample_rate)

    metadata = {'Selection': f"no-call_{idx}", 'Annotation': "No Call", 'Category': "no_calls", 'Spectrogram Shape': spectrogram_data.shape}
    
    # Save the no-call spectrogram with the 'no_calls' category
    np.savez(
        os.path.join(no_calls_output_dir, f"no_call_{idx}_spectrogram.npz"),
        spectrogram=spectrogram_data, metadata=metadata
    )

print(f"Saved spectrograms and metadata to {output_dir}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_25300\3696959251.py:2: UserWarning: nperseg = 2456 is greater than input length  = 2192, using nperseg = 2192
  frequencies, times, spectrogram_data = signal.spectrogram(


Saved spectrograms and metadata to spectrogram_output


### Bibliography

1. McKinney, W. (2010). Data Structures for Statistical Computing in Python. *Proceedings of the 9th Python in Science Conference*, 51-56. https://doi.org/10.25080/Majora-92bf1922-00

2. Brown, J.C. (1992). Computational Analysis of Sound Patterns. *Journal of the Acoustical Society of America*, 92(5), 2795-2798. https://doi.org/10.1121/1.407930

3. Oppenheim, A.V., & Schafer, R.W. (1989). *Discrete-Time Signal Processing*. Prentice Hall.

4. Stack Overflow. (2017). *How to convert a WAV file to a spectrogram in Python3*. Retrieved from https://stackoverflow.com/questions/44787437/how-to-convert-a-wav-file-to-a-spectrogram-in-python3

5. Stack Overflow. (2016). *How to create a numpy array from a pydub AudioSegment*. Retrieved from https://stackoverflow.com/questions/38015319/how-to-create-a-numpy-array-from-a-pydub-audiosegment

6. Stack Overflow. (2018). *How do I iterate over files in multiple directories in Python*. Retrieved from https://stackoverflow.com/questions/48133055/how-do-i-iterate-over-files-in-multiple-directories-in-python

7. Stack Overflow. (2017). *Adding together time duration columns in pandas*. Retrieved from https://stackoverflow.com/questions/44752208/adding-together-time-duration-columns-in-pandas

8. Stack Overflow. (2017). *How do I read a CSV file into a pandas dataframe*. Retrieved from https://stackoverflow.com/questions/46449764/how-do-i-read-a-csv-file-into-a-pandas-dataframe

9. Rajeev, R. (2021). *Spectrogram from audio data*. Retrieved from https://www.kaggle.com/code/rajeevratan84/spectrogram-from-audio-data

10. Sophia, G. (2021). *CNN for sound classification - Bird Calls 90%. Retrieved from https://www.kaggle.com/code/sophiagnetneva/cnn-for-sound-classification-bird-calls-90

11. Stack Overflow. (2018). *Extracting no-call segments from audio with gaps between call annotations*. Retrieved from https://stackoverflow.com/questions/50312258/how-do-i-split-a-long-waveform-with-annotated-periods-into-intervals-without-si

12. Stack Overflow. (2016). *How to segment audio based on annotation files?*. Retrieved from https://stackoverflow.com/questions/40011191/how-to-segment-a-wav-file-according-to-annotations-in-pandas

13. Kaggle. (2019). *How to segment data from audio into labeled intervals*. Retrieved from https://www.kaggle.com/competitions/tensorflow-speech-recognition-challenge/discussion/116233

14. Kaggle. (2021). *Handling and extracting segments from audio data*. Retrieved from https://www.kaggle.com/code/borisdayma/audio-steganography/data

15. Doshi, K. (2020). *Audio Deep Learning Made Simple (Part 1): State-of-the-Art Techniques*. Towards Data Science. Retrieved from https://towardsdatascience.com/audio-deep-learning-made-simple-part-1-state-of-the-art-techniques-742e62c29e53

16. Static Acoustic Monitoring of Harbour (Phoca vitulina) and Grey Seals (Halichoerus grypus) in the Malin Sea: A Revolutionary Approach in Pinniped Conservation. Retrieved from [journal website or link]

17. Stack Overflow. (2017). *Python - Break up a .wav file by timestamp*. Retrieved from https://stackoverflow.com/questions/44752208/python-break-up-a-wav-file-by-timestamp

18. Kaggle. (2021). *Bird song data set*. Retrieved from https://www.kaggle.com

19. Matplotlib. (2021). *Interactive Matplotlib: ipympl*. Retrieved from https://matplotlib.org/ipympl/